# SMARD Merit Order Analysis

## Table of Contents
1. [Setup and Imports](#setup-and-imports)
2. [Data Fetching](#data-fetching)
3. [Data Merging](#data-merging)
4. [Visualization and Feature Engineering](#visualization-and-feature-engineering)
5. [Regression Analysis](#regression-analysis)
6. [Machine Learning Models](#machine-learning-models)

## Setup and Imports

In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from functools import reduce
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression

# Unify file paths to be relative to the notebook directory
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')
CHUNKS_DIR = os.path.join(DATA_DIR, 'chunks')

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CHUNKS_DIR, exist_ok=True)
plt.close("all")
plt.rcParams.update({
    'figure.facecolor':  '#1e1e1e',
    'axes.facecolor':    '#1e1e1e',
    'axes.edgecolor':    '#555555',
    'axes.labelcolor':   '#cccccc',
    'axes.titlecolor':   '#ffffff',
    'xtick.color':       '#cccccc',
    'ytick.color':       '#cccccc',
    'grid.color':        '#333333',
    'grid.linestyle':    '--',
    'grid.alpha':        0.5,
    'text.color':        '#cccccc',
    'legend.facecolor':  '#2a2a2a',
    'legend.edgecolor':  '#555555',
    'figure.figsize':    (12, 6),
    'axes.spines.top':   False,
    'axes.spines.right': False,
})

## Data Fetching
Fetching available data chunks for SMARD filter IDs starting from Jan 1, 2016.

In [ ]:
START_MS = 1538344800000 # Jan 1 2016 UTC in milliseconds

SERIES = [
    (4169, "day_ahead_prices"),
    (410,  "grid_load"),
    (4068, "solar"),
    (1225, "wind_offshore"),
    (4067, "wind_onshore"),
]

def get_timestamps(filter_id, start_ms):
    url = f"https://www.smard.de/app/chart_data/{filter_id}/DE/index_hour.json"
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()
    return [ts for ts in data.get("timestamps", []) if ts >= start_ms]

def get_data(filter_id, timestamp):
    url = f"https://www.smard.de/app/chart_data/{filter_id}/DE/{filter_id}_DE_hour_{timestamp}.json"
    for attempt in range(5):
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            series = response.json().get("series", [])
            df = pd.DataFrame(series, columns=["timestamp", "value"])
            df["timestamp"] = (
                pd.to_datetime(df["timestamp"], unit="ms")
                .dt.tz_localize("UTC")
                .dt.tz_convert("Europe/Berlin")
            )
            return df
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(3)
    raise Exception(f"Failed to fetch {url} after 5 attempts")

def fetch_series(filter_id, name, start_ms):
    series_chunk_dir = os.path.join(CHUNKS_DIR, name)
    os.makedirs(series_chunk_dir, exist_ok=True)
    final_path = os.path.join(RAW_DIR, f"{name}.xlsx")
    
    if os.path.exists(final_path):
        print(f"[SKIP] {name} — final Excel already exists.")
        return

    timestamps = get_timestamps(filter_id, start_ms)
    total = len(timestamps)

    for i, ts in enumerate(timestamps):
        chunk_path = os.path.join(series_chunk_dir, f"{ts}.csv")
        if os.path.exists(chunk_path):
            continue
        print(f"[{name}] {i+1}/{total} fetching {ts}...")
        df_chunk = get_data(filter_id, ts)
        df_chunk.to_csv(chunk_path, index=False)

    all_chunks = [pd.read_csv(os.path.join(series_chunk_dir, f"{ts}.csv")) for ts in timestamps]
    df_final = pd.concat(all_chunks, ignore_index=True).dropna(subset=["value"]).reset_index(drop=True)
    df_final.to_excel(final_path, index=False)
    print(f"[{name}] Done. {len(df_final)} rows saved to {final_path}")

for filter_id, name in SERIES:
    fetch_series(filter_id, name, START_MS)

## Data Merging
Combines the raw Excel files into a single master DataFrame in memory.

In [ ]:
def load_and_rename(name, value_col_name):
    df = pd.read_excel(os.path.join(RAW_DIR, f"{name}.xlsx"))
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    df.rename(columns={'value': value_col_name}, inplace=True)
    return df

day_ahead = load_and_rename("day_ahead_prices", "dap")
grid_load = load_and_rename("grid_load", "gl")
solar = load_and_rename("solar", "slr")
wind_offshore = load_and_rename("wind_offshore", "wof")
wind_onshore = load_and_rename("wind_onshore", "won")

dataframes = [day_ahead, grid_load, solar, wind_offshore, wind_onshore]
merged = reduce(lambda left, right: pd.merge(left, right, on='timestamp', how='inner'), dataframes)

print(f"Rows before merge: {len(day_ahead)} DAP rows")
print(f"Rows after inner merge: {len(merged)}")

merged = merged.sort_values('timestamp').reset_index(drop=True)
merged['timestamp'] = merged['timestamp'].dt.tz_localize(None)

## Visualization and Feature Engineering
Plotting total renewables vs. day ahead prices, defining crisis periods, and establishing temporal features.

In [ ]:
os.makedirs('../figures', exist_ok=True)
os.makedirs('figures', exist_ok=True)
# Feature Engineering
merged['total_renewables'] = merged['slr'] + merged['won'] + merged['wof']
merged['is_negative'] = (merged['dap'] < 0).astype(int)

merged_resampled_anually = merged.resample('YE', on='timestamp')['is_negative'].sum()
merged_resampled_anually.index = merged_resampled_anually.index.year
merged_resampled_anually.plot.bar()
plt.savefig('../figures/negative_price_frequency.png', dpi=150, bbox_inches='tight', facecolor='#1e1e1e')

# Define Crisis Periods
conditions = [
    merged['timestamp'] < '2021-06-01',
    (merged['timestamp'] >= '2021-06-01') & (merged['timestamp'] < '2022-12-01'),
    merged['timestamp'] >= '2022-12-01'
]
choices = ['Pre-crisis', 'Crisis', 'Post-crisis']
merged['period'] = np.select(conditions, choices, default='Unknown')

# Plotting Day Ahead Price vs. Total Renewables by Period
plt.figure(figsize=(12, 8))
colors = {'Pre-crisis': '#4fc3f7', 'Crisis': '#ef5350', 'Post-crisis': '#66bb6a'}

for period in choices:
    subset = merged[merged['period'] == period]
    plt.scatter(
        subset['total_renewables'], 
        subset['dap'], 
        label=period, 
        color=colors[period], 
        alpha=0.1
    )

plt.title('Day Ahead Price vs. Total Renewables by Period')
plt.xlabel('Total Renewables')
plt.ylabel('Day Ahead Price')
legend = plt.legend(title='Period')
for handle in legend.legend_handles:
    handle.set_alpha(1.0)
plt.savefig('../figures/scatter_crisis.png', dpi=150, bbox_inches='tight', facecolor='#1e1e1e')
plt.show()

# Additional features for regression
merged['renewable_penetration'] = merged['total_renewables'] / merged['gl']
merged['hour'] = merged['timestamp'].dt.hour
merged['month'] = merged['timestamp'].dt.month
merged['day_of_week'] = merged['timestamp'].dt.dayofweek
merged['crisis_dummy'] = (merged['period'] == 'Crisis').astype(int)

## Regression Analysis
Analyzing the merit order effect, correcting for heteroskedasticity/autocorrelation with HAC.

In [ ]:
# Model 1: Initial model dropping Grid Load to prevent multicollinearity (Logic behind which varibles to drop explained in regression.ipynb)
model_base = smf.ols(
    formula='dap ~ renewable_penetration + C(hour) + C(month) + C(day_of_week) + crisis_dummy', 
    data=merged
)
res_base = model_base.fit(cov_type='HAC', cov_kwds={'maxlags' : 24})
print("BASE MODEL SUMMARY\n", res_base.summary())

# Split pre and post crisis
pre = merged[merged['period'] == 'Pre-crisis']
post = merged[merged['period'] == 'Post-crisis']

model_pre = smf.ols(formula='dap ~ renewable_penetration + C(hour) + C(month) + C(day_of_week)', data=pre)
res_pre = model_pre.fit(cov_type='HAC', cov_kwds={'maxlags' : 24})
print("\nPRE-CRISIS SUMMARY\n", res_pre.summary())

model_post = smf.ols(formula='dap ~ renewable_penetration + C(hour) + C(month) + C(day_of_week)', data=post)
res_post = model_post.fit(cov_type='HAC', cov_kwds={'maxlags' : 24})
print("\nPOST-CRISIS SUMMARY\n", res_post.summary())

## Machine Learning Models
Predicting negative price events (`is_negative`) using Logistic Regression and Gradient Boosting.

In [ ]:
FEATURES = ['renewable_penetration', 'hour', 'month', 'day_of_week', 'slr', 'won', 'wof']
TARGET = 'is_negative'

train = merged[merged['timestamp'] < '2025-01-01']
test  = merged[merged['timestamp'] >= '2025-01-01']

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"Training hours: {len(train)}")
print(f"Test hours: {len(test)}")
print(f"Negative hours in test: {y_test.sum()} ({100*y_test.mean():.1f}%)\n")

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
print("LOGISTIC REGRESSION BASELINE")
print(classification_report(y_test, lr_preds))

# Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
gb.fit(X_train, y_train)
gb_preds = gb.predict(X_test)
print("GRADIENT BOOSTING")
print(classification_report(y_test, gb_preds))

# Feature Importances Plot
importances = gb.feature_importances_
feat_df = pd.DataFrame({'feature': FEATURES, 'importance': importances}).sort_values('importance', ascending=True)
feat_df.plot.barh(x='feature', y='importance', title='Feature Importances')
plt.tight_layout()
plt.savefig('../figures/feature_importances.png', dpi=150, bbox_inches='tight', facecolor='#1e1e1e')
plt.show()

merged.to_parquet(os.path.join(DATA_DIR, "final_merged_data.parquet"), index=False)